# Vietnamese TTS: 4 Approaches Compared

## The Problem
Mainstream TTS models (Parler-TTS, Bark, XTTS-v2) have poor or no Vietnamese support.

## Vietnamese TTS Landscape (2026)

| Model | Voice Cloning | Quality | Speed | GPU Required |
|-------|---------------|---------|-------|-------------|
| **viXTTS** (`thinhlpg/viXTTS`) | Yes (6-30s ref) | High | Slow | Yes |
| **F5-TTS Vietnamese** (`hynt/F5-TTS-Vietnamese-ViVoice`) | Yes (3-10s ref) | High | Medium | Yes |
| **Edge TTS** (Microsoft) | No (4 preset voices) | Very High | Fast | No (cloud) |
| **Facebook MMS-TTS** (`facebook/mms-tts-vie`) | No (single speaker) | Decent | Fast | Optional |
| **VieNeu-TTS** (`pip install vieneu`) | Yes (3-5s ref) | High | Fast | Optional |
| **MeloTTS Vietnamese** | No | Good | Fast | Yes |
| **Kokoro / Orpheus** | — | — | — | **No Vietnamese** |

This notebook tests the **top 4** approaches side by side.

---
**Kaggle Setup**: GPU T4 x2 | Internet enabled

## Step 1: Install Dependencies (~5 min)

In [ ]:
%%time
import os
os.environ["COQUI_TOS_AGREED"] = "1"

# Step 1: Install coqui-tts
!pip install -q coqui-tts

# Step 2: Restore 'imp' module shim (removed in Python 3.12, needed by vinorm)
# vinorm uses: imp.find_module('vinorm') to get the package directory
# Returns (file, pathname, description) — pathname must be the DIRECTORY
import importlib, importlib.util, types, sys
imp = types.ModuleType('imp')
imp.reload = importlib.reload
def _find_module(name, path=None):
    spec = importlib.util.find_spec(name)
    if spec is None:
        raise ImportError(f"No module named {name}")
    # Return package directory (not file path) — vinorm writes files there
    origin = spec.origin or ""
    pkg_dir = os.path.dirname(origin) if origin else ""
    return (None, pkg_dir, ('', '', 6))
imp.find_module = _find_module
sys.modules['imp'] = imp

# Step 3: Install Vietnamese text processing deps
!pip install -q vinorm==2.0.7 cutlet underthesea unidic-lite
!pip install -q --force-reinstall 'protobuf>=5.26.1'

# Step 4: Monkey-patch Vietnamese language support into coqui-tts
import TTS.tts.layers.xtts.tokenizer as xtts_tokenizer
import TTS.tts.configs.xtts_config as xtts_config

_orig_tokenizer_init = xtts_tokenizer.VoiceBpeTokenizer.__init__
def _patched_tokenizer_init(self, *args, **kwargs):
    _orig_tokenizer_init(self, *args, **kwargs)
    self.char_limits["vi"] = 250
xtts_tokenizer.VoiceBpeTokenizer.__init__ = _patched_tokenizer_init

_orig_preprocess = xtts_tokenizer.VoiceBpeTokenizer.preprocess_text
def _patched_preprocess(self, txt, lang):
    if lang == "vi":
        return txt.lower().strip()
    return _orig_preprocess(self, txt, lang)
xtts_tokenizer.VoiceBpeTokenizer.preprocess_text = _patched_preprocess

_orig_config_init = xtts_config.XttsConfig.__init__
def _patched_config_init(self, **kwargs):
    _orig_config_init(self, **kwargs)
    if "vi" not in self.languages:
        self.languages.append("vi")
xtts_config.XttsConfig.__init__ = _patched_config_init

print("coqui-tts installed and Vietnamese language support patched!")

In [ ]:
# Download pre-trained viXTTS model from HuggingFace
from huggingface_hub import snapshot_download

print("Downloading viXTTS model (~1.8GB)...")
snapshot_download(
    repo_id="thinhlpg/viXTTS",
    repo_type="model",
    local_dir="model"
)
print("Model downloaded!")

# Check downloaded files
import os
for f in sorted(os.listdir("model")):
    fpath = f"model/{f}"
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1e6
        print(f"  {f}: {size:.1f} MB")

## Step 2: Load Model

In [ ]:
import torch
import torchaudio
import time
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
from IPython.display import Audio, display

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load viXTTS model
print("Loading viXTTS model...")
t0 = time.time()

config = XttsConfig()
config.load_json("model/config.json")

model = Xtts.init_from_config(config)
model.load_checkpoint(
    config,
    checkpoint_path="model/model.pth",
    vocab_path="model/vocab.json",
    use_deepspeed=False,  # set True if you installed deepspeed
)
model.to(device)

print(f"Model loaded in {time.time()-t0:.1f}s")

## Step 3: Vietnamese Text Normalization

Vietnamese TTS needs special text preprocessing:
- `20/11` → `hai mươi tháng mười một`
- `AI` → `Ây Ai`
- `100.000đ` → `một trăm nghìn đồng`
- Handle diacritics: `ă, â, ơ, ư, đ, ...`

This is handled by `vinorm` (Vietnamese Text Normalization).

In [ ]:
from vinorm import TTSnorm
from underthesea import sent_tokenize

# Demo: Vietnamese text normalization
examples = [
    "Hôm nay là ngày 20/11, nhiệt độ ngoài trời là 35°C.",
    "Công ty ABC có doanh thu 100.000.000đ trong Q3/2024.",
    "AI và Machine Learning đang thay đổi thế giới.",
    "Địa chỉ: 123 Nguyễn Huệ, Q.1, TP.HCM.",
]

print("Vietnamese Text Normalization Examples:")
print("=" * 60)
for text in examples:
    normalized = TTSnorm(text, unknown=False, lower=False, rule=True)
    print(f"\nOriginal:   {text}")
    print(f"Normalized: {normalized}")

## Step 4: Generate Vietnamese Speech

The model comes with pre-built Vietnamese voice samples:
- `nam-calm.wav` — Male, calm
- `nam-truyen-cam.wav` — Male, expressive
- `nu-calm.wav` — Female, calm
- `nu-nhe-nhang.wav` — Female, gentle
- `nu-luu-loat.wav` — Female, fluent

In [ ]:
def normalize_vietnamese(text):
    """Normalize Vietnamese text for TTS."""
    text = (
        TTSnorm(text, unknown=False, lower=False, rule=True)
        .replace("..", ".")
        .replace("!.", "!")
        .replace("?.", "?")
        .replace(" .", ".")
        .replace(" ,", ",")
        .replace('"', "")
        .replace("'", "")
        .replace("AI", "Ây Ai")
        .replace("A.I", "Ây Ai")
    )
    return text


def speak_vietnamese(model, text, speaker_wav, normalize=True, device="cuda:0"):
    """Generate Vietnamese speech with voice cloning."""
    # Get speaker embedding from reference audio
    gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(
        audio_path=speaker_wav,
        gpt_cond_len=model.config.gpt_cond_len,
        max_ref_length=model.config.max_ref_len,
        sound_norm_refs=model.config.sound_norm_refs,
    )
    
    # Normalize text
    if normalize:
        text = normalize_vietnamese(text)
    
    # Split into sentences
    sentences = sent_tokenize(text)
    
    wav_chunks = []
    for sent in sentences:
        if not sent.strip():
            continue
        wav_chunk = model.inference(
            text=sent,
            language="vi",
            gpt_cond_latent=gpt_cond_latent,
            speaker_embedding=speaker_embedding,
            temperature=0.3,
            length_penalty=1.0,
            repetition_penalty=10.0,
            top_k=30,
            top_p=0.85,
        )
        wav_chunks.append(torch.tensor(wav_chunk["wav"]))
    
    out_wav = torch.cat(wav_chunks, dim=0).unsqueeze(0)
    return out_wav

print("Vietnamese TTS function ready!")

In [ ]:
# List available voice samples
import glob
samples = sorted(glob.glob("model/samples/*.wav"))
print("Available Vietnamese voice samples:")
for s in samples:
    print(f"  🎙️ {os.path.basename(s)}")
    display(Audio(s))

In [ ]:
# Generate Vietnamese speech!
text = "Xin chào! Tôi là trợ lý giọng nói tiếng Việt. Tôi có thể đọc bất kỳ văn bản nào bằng giọng của bạn."

# Try with female calm voice
speaker_wav = "model/samples/nu-calm.wav"
print(f"Speaker: {os.path.basename(speaker_wav)}")
print(f"Text: {text}")

t0 = time.time()
audio = speak_vietnamese(model, text, speaker_wav, device=device)
print(f"\nGenerated in {time.time()-t0:.1f}s")

display(Audio(audio, rate=24000))
torchaudio.save("vi_output_1.wav", audio, 24000)

## Step 5: Compare Different Vietnamese Voices

In [ ]:
text = "Trí tuệ nhân tạo đang thay đổi cách chúng ta sống và làm việc mỗi ngày."

voice_samples = {
    "Nam - Bình tĩnh": "model/samples/nam-calm.wav",
    "Nam - Truyền cảm": "model/samples/nam-truyen-cam.wav",
    "Nữ - Bình tĩnh": "model/samples/nu-calm.wav",
    "Nữ - Nhẹ nhàng": "model/samples/nu-nhe-nhang.wav",
}

for name, wav_path in voice_samples.items():
    if os.path.exists(wav_path):
        print(f"\n{'='*50}")
        print(f"🎙️ {name}")
        t0 = time.time()
        audio = speak_vietnamese(model, text, wav_path, device=device)
        print(f"Generated in {time.time()-t0:.1f}s")
        display(Audio(audio, rate=24000))
    else:
        print(f"\n⚠️ {name}: file not found at {wav_path}")

## Step 6: Long Text — Vietnamese Article Reading

In [ ]:
article = """
Việt Nam là một quốc gia nằm ở phía đông bán đảo Đông Dương thuộc khu vực Đông Nam Á.
Với dân số khoảng 100 triệu người, Việt Nam là quốc gia đông dân thứ 15 trên thế giới.
Thủ đô Hà Nội là trung tâm chính trị và văn hóa của cả nước.
Thành phố Hồ Chí Minh là thành phố lớn nhất và là trung tâm kinh tế quan trọng nhất.
""".strip()

print("Generating Vietnamese article reading...")
print(f"Text length: {len(article)} chars")

speaker_wav = "model/samples/nam-truyen-cam.wav"
t0 = time.time()
audio = speak_vietnamese(model, article, speaker_wav, device=device)
print(f"\nGenerated in {time.time()-t0:.1f}s")
print(f"Audio duration: {audio.shape[1]/24000:.1f}s")

display(Audio(audio, rate=24000))
torchaudio.save("vi_article.wav", audio, 24000)
print("Saved to vi_article.wav")

## Step 7: Voice Cloning — Use Your Own Voice

Upload a WAV file of your voice (6-30 seconds, clean recording) and the model will speak in your voice!

In [ ]:
# To clone your own voice:
# 1. Upload a WAV file to Kaggle (Settings → Data → Upload)
# 2. Update the path below

# my_voice = "/kaggle/input/my-voice/recording.wav"
# text = "Đây là giọng nói của tôi được nhân bản bởi trí tuệ nhân tạo."
# audio = speak_vietnamese(model, text, my_voice, device=device)
# display(Audio(audio, rate=24000))

print("To clone your voice:")
print("1. Record 10-30s of clear Vietnamese speech")
print("2. Save as WAV (22050 Hz, mono)")
print("3. Upload as Kaggle dataset")
print("4. Uncomment the code above and update the path")

## Key Takeaways

### Vietnamese TTS — Full Landscape

| Model | Voice Cloning | Quality | GPU | Install |
|-------|---------------|---------|-----|---------|
| **viXTTS** | Yes (6-30s) | High | Yes | git clone + HF model |
| **F5-TTS Vietnamese** | Yes (3-10s) | High | Yes | `pip install f5-tts` + HF checkpoint |
| **VieNeu-TTS** | Yes (3-5s) | High | Optional | `pip install vieneu` |
| **Edge TTS** | No (4 voices) | Very High | No | `pip install edge-tts` |
| **Facebook MMS** | No (1 voice) | Decent | No | `pip install transformers` |
| **VietTTS** | Yes | High | Yes | `pip install viet-tts` |
| **MeloTTS Vietnamese** | No | Good | Yes | git clone |

### Recommendations:
- **Best quality, no cloning needed**: Edge TTS (Microsoft HoaiMy/NamMinh)
- **Voice cloning needed**: viXTTS or F5-TTS Vietnamese (ViVoice 1000h)
- **Lightweight / CPU**: Facebook MMS-TTS (`facebook/mms-tts-vie`)
- **Newest, Vietnamese-native**: VieNeu-TTS (`pip install vieneu`)

### Key Vietnamese-specific components:
- **`vinorm`** — text normalization (numbers, dates, abbreviations → spoken form)
- **`underthesea`** — sentence tokenization for Vietnamese
- Vietnamese has **6 tones** — models must handle tonal marks correctly

### Models that do NOT support Vietnamese:
- Kokoro TTS (requested but not planned)
- Orpheus TTS (English-focused)
- Parler-TTS (English only)
- XTTS-v2 base (use viXTTS fork instead)

In [ ]:
!pip install -q edge-tts

import edge_tts
import asyncio

async def edge_tts_generate(text, voice, output_file):
    """Generate Vietnamese speech using Microsoft Edge TTS."""
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(output_file)

# Test both Vietnamese voices
test_text = "Xin chào! Tôi là trợ lý giọng nói tiếng Việt của Microsoft. Tôi có thể đọc bất kỳ văn bản nào."

voices = [
    ("vi-VN-HoaiMyNeural", "Female (HoaiMy)"),
    ("vi-VN-NamMinhNeural", "Male (NamMinh)"),
]

for voice_id, label in voices:
    out_file = f"edge_tts_{voice_id}.mp3"
    await edge_tts_generate(test_text, voice_id, out_file)
    print(f"\n🎙️ {label} ({voice_id})")
    display(Audio(out_file))

---

## Approach 3: Facebook MMS-TTS — Lightweight, HuggingFace Native

Facebook's Massively Multilingual Speech project trained VITS checkpoints for 1100+ languages.
Vietnamese model: `facebook/mms-tts-vie`

- No voice cloning (single speaker)
- Runs on CPU or GPU
- Simple HuggingFace Transformers API

In [ ]:
from transformers import VitsModel, AutoTokenizer
import scipy.io.wavfile as wavfile
import numpy as np

# Load Facebook MMS Vietnamese model
print("Loading facebook/mms-tts-vie...")
mms_model = VitsModel.from_pretrained("facebook/mms-tts-vie")
mms_tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-vie")
print(f"Loaded! Sample rate: {mms_model.config.sampling_rate} Hz")

# Generate speech
mms_texts = [
    "Xin chào! Đây là mô hình chuyển đổi văn bản thành giọng nói của Facebook.",
    "Việt Nam là một quốc gia xinh đẹp với nền văn hóa đa dạng và phong phú.",
]

for i, text in enumerate(mms_texts):
    inputs = mms_tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        output = mms_model(**inputs).waveform
    
    audio_np = output[0].numpy()
    out_file = f"mms_vi_{i}.wav"
    wavfile.write(out_file, rate=mms_model.config.sampling_rate, data=audio_np)
    
    print(f"\n📝 {text}")
    display(Audio(out_file))